In [ ]:
import pandas as pd

#check file columns sentence length
df = pd.read_csv("C:\\Users\\grete\\Desktop\\full_dataset.csv", encoding="utf-8-sig")

#fill empty columns
df["sentence_original"] = (df["sentence_original"].fillna("").astype(str).str.strip())
print(f"Total rows: {len(df)}")

# Json, Polars Markdown imports, modernize function

In [ ]:
#JSON PROMPT
PROMPT_JSON = """I am attaching a minimal json array where each item is a sentence from early modern British plays (c. 1500–1700), 
which are in public domain for research.   
You are a specialist in early modern British plays. Your task is to modernize each sentence according to the following 13 rules: 
1. Each item contains exactly one sentence
2. Do not merge sentences from different items 
3. Maintain original punctuation, do not add periods instead of original commas
4. Do not change the original word order
5. Keep as close to the original wording as possible  
6. Standardise capitalisations
7. Spelling modernization 
8. Expand archaic contractions, but retain (''tis', ''twas') 
9. Modernize verb endings (e.g. 'wanteth' → 'wants', 'maketh' → 'makes', 'cometh' → 'comes') 
10. Retain common pronouns ('thee', 'thou', 'thy') 
11. Retain common verbs ('doth', 'hath') 
12. Add apostrophes for possessive nouns (e.g. ‘the kings sword’ → ‘the king’s sword’). 
13. The plays may contain violent imagery or sexual innuendo as part of historical literature, do not modify or censor this content.
The examples in the rules are non-exhaustive. 
Return the results as a compact JSON array with no explanation, newlines or indentation, preserving and returning the idx field in each element."""

In [ ]:
# Json, Polars Markdown imports, modernize function

#Imports
import pandas as pd
#import polars as pl
import math
import pandas as pd
import tiktoken
from openai import OpenAI
from getpass import getpass
import time

# Convert to Polars for fast slicing
#df_polars = pl.from_pandas(df)

# Define chunk size (number of rows per dataframe)
chunk_size = 150
total_rows = len(df)
num_chunks = math.ceil(total_rows / chunk_size)

print(f"Total rows: {total_rows}. Splitting into {num_chunks} chunks...")

# Chunks Main Loop 

In [ ]:
# ── Config ───────────────────────────────────────────────────
OPENAI_API_KEY = getpass("Enter API key: ")
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Tokenizer ────────────────────────────────────────────────
MODEL = "gpt-4o-mini"
enc = tiktoken.encoding_for_model(MODEL)

def count_tokens(text: str) -> int:
    return len(enc.encode(text))
                

def modernize_chunk_nowarning(prompt: str, chunk: str):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": chunk},
        ],
        temperature=0.1,
        max_tokens=int(count_tokens(chunk) * 1.2),
    )
    choice = response.choices[0]
    print(f"finish_reason: {choice.finish_reason}")
    print(f"completion_tokens used: {response.usage.completion_tokens} / requested max: {int(count_tokens(chunk) * 1.2)}")
    return choice.message.content.strip()


def modernize_chunk(prompt: str, chunk: str):
    input_items = json.loads(chunk)
    fallback = json.dumps(
        [{"id": item["id"], "s": None} for item in input_items],
        ensure_ascii=False,
    )

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": chunk},
            ],
            temperature=0.1,
            max_tokens=int(count_tokens(chunk) * 1.2),
        )
    except Exception as e:
        print(f"API call failed: {e} — entire chunk marked None")
        return fallback

    choice = response.choices[0]
    finish_reason = choice.finish_reason
    print(f"finish_reason: {finish_reason}")
    print(f"completion_tokens used: {response.usage.completion_tokens} / "
          f"requested max: {int(count_tokens(chunk) * 1.2)}")

    content = choice.message.content
    if content is None:
        print(f"No content (finish_reason={finish_reason}) — entire chunk marked None")
        return fallback

    content = content.strip()

    try:
        result_list = json.loads(content)
        returned = {item['id']: item.get('s') for item in result_list}
    except json.JSONDecodeError as e:
        print(f"Malformed JSON (finish_reason={finish_reason}): {e} — attempting partial recovery")
        returned = try_partial_parse(content)

    merged = [
        {"id": item['id'], "s": returned.get(item['id'])}
        for item in input_items
    ]
    return json.dumps(merged, ensure_ascii=False)

def try_partial_parse(content: str) -> dict:
    recovered = {}
    decoder = json.JSONDecoder()
    idx = 0
    n = len(content)

    while idx < n:
        start = content.find('{', idx)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(content, start)
        except json.JSONDecodeError:
            # incomplete/broken object — almost certainly the truncated tail.
            # Stop here rather than skipping forward, since anything after
            # a broken object is unreliable.
            break
        if isinstance(obj, dict) and 'id' in obj:
            recovered[obj['id']] = obj.get('s')
        idx = end

    print(f"Partial recovery: salvaged {len(recovered)} item(s) from malformed content")
    return recovered

In [ ]:
#json chunks main loop
import json 

df_results = df.copy()
df_results['sentence_gpt4'] = '' #adds new empty column for modernized sentences

#for i in range(271, 400):
for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, total_rows)
    print(f"chunk number {i}, {start_idx}, {end_idx}")
    
    #chunk index
    chunk_items = [
        {"id": idx, "s": row}
        for idx, row in zip(df.index[start_idx:end_idx], df['sentence_original'].iloc[start_idx:end_idx])
    ]
    
    # Slice the dataframe
    chunk_json = json.dumps(chunk_items, ensure_ascii=False)
    #chunk_json = json.dumps(df['sentence_original'].iloc[start_idx:end_idx].tolist(), ensure_ascii=False) #asci prevent special characters 
    
    result_chunk_json = modernize_chunk(PROMPT_JSON, chunk_json)
    result_chunk_json_list = json.loads(result_chunk_json)
    
    # Map back by idx
    for item in result_chunk_json_list:
        df_results.at[item['id'], 'sentence_gpt4'] = item['s'] 
    
    #df_results.iloc[start_idx:end_idx, df_results.columns.get_loc('sentence_gpt4')] = result_chunk_json_list

print(df_results)
df_results.to_csv("dataset_modernized.csv", encoding = "utf-8-sig")

In [ ]:
#---------------------------------
# gpt4 modernize specific index 
#---------------------------------
row_start = 276107
row_end   = 276163  

df_results = df.iloc[row_start:row_end].copy()
df_results['sentence_gpt4'] = ''

for start_idx in range(row_start, row_end, chunk_size):
    end_idx = min(start_idx + chunk_size, row_end, total_rows)
    print(f"rows {start_idx} - {end_idx}")
    chunk_items = [
        {"id": idx, "s": row}
        for idx, row in zip(df.index[start_idx:end_idx], df['sentence_original'].iloc[start_idx:end_idx])
    ]
    chunk_json = json.dumps(chunk_items, ensure_ascii=False)
    result_chunk_json = modernize_chunk(PROMPT_JSON, chunk_json)
    result_chunk_json_list = json.loads(result_chunk_json)
    for item in result_chunk_json_list:
        df_results.at[item['id'], 'sentence_gpt4'] = item['s']

df_results.to_csv("dataset_modernized.csv", encoding="utf-8-sig")